# Recompute `YrA` from the saved footprints

Driver for `caban.yra_recompute`. All the logic lives in `caban/yra_recompute.py`;
this notebook is the parameter cell, the queue, and the report
(`plans/yra_recompute_plan.md` §11.1).

**How to use it:** run the setup and discovery cells, then the **batch** cell: it works
through every session in the queue unattended, printing each session's report as it
goes. Finished sessions are recognised from what is on disk (the output is still there
and its inputs still hash the same), so after a stop or a kernel restart, re-running the
batch cell prints their reports again and carries on with the first unfinished one.

To go one session at a time instead — e.g. to look at a problem session — use the
optional **preview** and **run** cells at the end.

**Order of execution** (§11.7): G10 `TFC_cond` was the §7.1 gate and has passed
(§12). The queue below picks up from there.

`YrA_recomputed.zarr` and its `YrA_recompute.json` sidecar are written beside the
`A`/`C`/`S` they come from, inside the session's `minian_crossreg*` folder; the existing
`YrA.zarr` there is never touched (§11.3).

In [ ]:
import os
import sys
import time

import dask
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from caban import minian_runner as mr
from caban import yra_recompute as yr

pd.set_option("display.max_rows", 200, "display.width", 220)

## Parameters

Nothing here is per-session. Every path a session needs — its videos, its
`minian_crossreg*`, its old `YrA.zarr` if it has one, its saved `Y` if it has one —
is resolved by discovery, and `del_frames` is looked up from the notebook corpus
(see `yr.NOTEBOOK_DEL_FRAMES`: across all 60 notebooks in the Minian mirror the only
session with frames dropped is a G25 one, outside this cohort). The derivation from
the saved `frame` coordinate still runs as an independent cross-check, and a
disagreement is a hard failure (§6.1).

`MINIAN_DIR_OVERRIDES` exists because discovery refuses to guess. G16's
`2022_01_26-TFC_test_B / 14_23_45-LT1` has two complete `minian_crossreg*` outputs
that disagree about both unit and frame count, and only one of them can belong to
this session: the videos and `timeStamps.csv` hold 13,923 frames, which is
`minian_crossreg3` (788 x 13923). `minian_crossreg2_crossreg3` is (883 x 17853) —
more frames than the session has, so it belongs to something else. Hence the
override below.

Picking wrong is not silently possible: `derive_deleted_frames` raises on a `frame`
coordinate that runs past the end of the videos, which is exactly what the other
folder does. The override says which one is meant; the guard enforces that it fits.

In [ ]:
DATA_ROOTS = None                   # auto: /Volumes/MINISCOPE if mounted, else the
                                    # two read-only BAK drives

MINIAN_DIR_OVERRIDES = {
    "G16/2022_01_26-TFC_test_B/14_23_45-LT1": "minian_crossreg3",
}

# Narrow the queue while working through a mouse or a session type. None = everything.
MOUSE_FILTER = None                 # e.g. ["G05", "G08"]
SESSION_FILTER = None               # substring of the session name, e.g. "TFC_cond"

N_WORKERS = 6
FRAME_CHUNK = yr.DEFAULT_FRAME_CHUNK
FORCE = False                       # recompute even a session whose sidecar says done

dask.config.set(scheduler="threads", num_workers=N_WORKERS)

## Discover the queue

A session is work exactly when its `Miniscope` folder has numbered `.avi` files and
one complete `minian_crossreg*`. The counts printed below account for everything
scanned, so what is *not* in the queue is visible rather than implied — the ~976
excluded folders are the HC and CNO sessions that were never put through CNMF-E.

The table is the queue. `done` comes from a cheap check (a complete sidecar with its
output beside it); the authoritative check re-hashes every input and runs inside
`run_item`, so a session that merely looks finished is still caught.

In [ ]:
items = yr.discover_sessions(
    roots=DATA_ROOTS,
    minian_dir_overrides=MINIAN_DIR_OVERRIDES,
)

if MOUSE_FILTER:
    items = [i for i in items if i.mouse in MOUSE_FILTER]
if SESSION_FILTER:
    items = [i for i in items if SESSION_FILTER in i.session]

status = yr.queue_status(items)
print(f"\n{int(status['done'].sum())} of {len(status)} done")
status

## Run the queue unattended

Serial by design (§11.5) — each session is already memory- and I/O-bound through its own
dask graph. Roughly a day for all 130: ~11 min for a 26-file conditioning session at
`N_WORKERS = 6`, less for the shorter LT sessions; about half of it is decoding `.avi` over
USB. Each session writes `YrA_recomputed.zarr` plus its `YrA_recompute.json` sidecar, then
prints its report.

`stop_on_error=False`: a session that fails a hard check is recorded and the queue moves
on; the failures are listed at the end. Two are known — G05 `2021_09_01-TFC_test_B`
`15_37_05-LT1` and `16_20_07-TFC_test_B`, whose saved `motion.zarr` is NaN in every frame
(§14). Set it to `True` to stop at the first failure instead.

**Reading a report:** hard assertions run inside the call, so reaching the report means
`unit_id` matches `S`'s elementwise and unsorted, shape and `frame` match `C`, and there are
no NaN or inf values. Then read the `overlap none` row first. Those units have `YrA_i`
determined by the replayed movie and their own footprint alone — no `C` at all — so a
median r that is not ~0.999999 means that session's `Y` did not replay. The `>10 %` row
sitting lower is the known `C`-version difference of §12.4, not a replay problem. Sessions
with no old `YrA` have no comparison block at all and lean on the hard checks plus
`corr(C, YrA_new)`.

In [ ]:
records = yr.run_all(items, frame_chunk=FRAME_CHUNK, force=FORCE, stop_on_error=False)

failed = [r for r in records if "failed" in r]
print(f"\n{len(records) - len(failed)} of {len(records)} sessions done; {len(failed)} failed")
for r in failed:
    print(f"  {r['label']}: {r['failed']}")
yr.queue_status(items)

## Re-check every replay against production's `max_proj`

`plans/yra_recompute_plan.md` §14.2. The old-`YrA` comparison proves the replay only for the
28 sessions that had a `YrA` exported; this proves it for every session. It replays the movie
alone (no `YrA`) and compares its maximum over frames with the `max_proj` production saved.
A pixel off by more than 1 grey level is a hard failure: the movie is not production's.
Pixels off by exactly 1 (caban's cv2 vs 2021's) are counted in `max_proj_n_differing`, not
bounded, until this run has calibrated them.

`YrA_recomputed.zarr` is never rewritten; the result goes into each sidecar as
`report.max_proj_check`, failures included. Sessions already checked and passing are skipped,
so re-running this cell resumes. Sessions recomputed since the check was built were checked as
part of the recompute and are skipped here too. The two G05 NaN-motion sessions fail at once,
as in the batch.

Reads every session's videos, so about as long as the batch minus the `YrA` arithmetic: run it
where the data is local (the Razer, after the MINIRAZER copy).

In [ ]:
records = yr.verify_all(items, frame_chunk=FRAME_CHUNK, stop_on_error=False)

failed = [r for r in records if "failed" in r]
print(f"\n{len(records) - len(failed)} of {len(records)} replays match max_proj; {len(failed)} failed")
for r in failed:
    print(f"  {r['label']}: {r['failed']}")
yr.queue_status(items)

## Recover the two G05 NaN-motion sessions

`plans/yra_recompute_plan.md` §14.1. G05 `2021_09_01-TFC_test_B` `15_37_05-LT1` and
`16_20_07-TFC_test_B` have a saved `motion.zarr` that is NaN in every frame, so the batch cannot
recompute their `YrA`. Per session:

1. `mr.reestimate_motion` runs the Minian notebook up to `max_proj` only (no CNMF), all output
   in scratch on FUTROLA, and requires its `max_proj` to equal production's **exactly**. Only
   then does it write `motion_reestimated.zarr` (+ `.json`) into the production folder. On a
   mismatch it stops, writes nothing there and keeps the scratch for inspection.
2. `yr.run_item` recomputes `YrA` with it; it picks `motion_reestimated` up by itself.

About 10–20 min for step 1 per session, plus the recompute. Needs the MINIRAZER copy finished,
like the cell above. A session already done in step 1 raises `FileExistsError` — skip to step 2.

In [ ]:
g05_nan_motion = [i for i in items
                  if i.label.endswith(("15_37_05-LT1", "16_20_07-TFC_test_B")) and i.mouse.startswith("G05")]
assert len(g05_nan_motion) == 2, [i.label for i in g05_nan_motion]

for item in g05_nan_motion:
    print(f"\n=== {item.label}")
    mr.reestimate_motion(item, n_workers=N_WORKERS)
    yr.run_item(item, frame_chunk=FRAME_CHUNK)

## Optional: one session at a time

For looking at a single session — the next one not yet done — instead of the batch.
**Preview** reports its resolved paths, the unit and frame counts, and the two `del_frames`
values side by side, then stops before the replay. **Run** does the work and prints the
report (see *Reading a report* above). Preview again and it has advanced to the next session.

**Commented out** so that *Run All* runs only the sections above. Uncomment both cells to use them.

In [ ]:
# item = yr.next_pending(items)
# if item is None:
#     print("queue empty -- every session in this selection is done")
# else:
#     _ = yr.preview_item(item, frame_chunk=FRAME_CHUNK)

In [ ]:
# started = time.time()
# record = yr.run_item(item, frame_chunk=FRAME_CHUNK, force=FORCE)
# print(f"\nelapsed {time.time() - started:.1f} s")